# Clase 07 — Análisis gramatical y entidades

**Curso:** NLP Clásico con Python · **Bloque 3:** Estructura y significado del texto
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/07_analisis_gramatical_entidades.md`

## Qué vas a ver en esta clase

- **POS** (categoría gramatical) y **dependencias sintácticas** con spaCy en español y en italiano: leer una tabla de tokens y un árbol.
- **Extraer** «quién hace qué» (sujeto, verbo, objeto) recorriendo el árbol, y ver dónde falla un modelo pequeño.
- **NER**: entidades nombradas (PER, ORG, LOC, MISC) sobre un corpus de 67 frases de noticias sintéticas anotadas a mano por idioma.
- **Evaluar** un NER: definir antes qué es un acierto (tramo y etiqueta exactos frente a coincidencia parcial), calcular precisión, *recall* y F1 **a mano** y por entidad.
- **Errores típicos** de los modelos pequeños y **reglas** (`Matcher`, `EntityRuler`) para corregirlos, midiendo en frases que no se usaron para escribirlas.

## Requisitos

- Tokenización y lematización (clases 2 y 3); precisión, *recall* y F1 como ideas (clase 5 las retoma con más detalle).
- Python con `numpy`, `pandas`, `matplotlib` y `spacy` 3.8 con `es_core_news_sm` y `it_core_news_sm`. La celda siguiente instala solo lo que falte (también en Google Colab).
- La carpeta `datos/` del curso (`noticias_ner_es.csv` y `noticias_ner_it.csv`). En Colab, súbela desde el panel de archivos.

> **Cómo usar este notebook:** ejecuta las celdas en orden. Las predicciones de spaCy dependen de la versión del modelo (aquí, spaCy 3.8 con los modelos 3.8.0): las cifras son un *ejemplo ilustrativo* sobre un corpus pequeño y no se generalizan. Los cálculos que dependen solo de nuestro código se comprueban con `assert`.

---
## Instalación

Instala solo lo que falte (paquetes y modelos). Si ya está todo, no hace nada.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("spacy", "spacy")
print("Librerías listas.")

---
## Celda 0 — Setup

Importamos las librerías, fijamos la **semilla**, definimos `cargar_datos` y `cargar_modelo` y cargamos los dos modelos pequeños. Anota las versiones que se imprimen: las predicciones que veas dependen de ellas.

In [ ]:
import os
import re
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42)                          # semilla fija
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

import spacy

MODELOS = {"spa": "es_core_news_sm", "ita": "it_core_news_sm"}

def cargar_modelo(nombre):
    """Carga un modelo de spaCy y lo descarga si falta (idempotente)."""
    try:
        return spacy.load(nombre)
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", "-q", nombre], check=True)
        return spacy.load(nombre)

MARCA = re.compile(r"\{([^{}|]+)\|(PER|ORG|LOC|MISC)\}")

def parsear(marcado):
    """Convierte 'Hola {Ana|PER}' en (texto, {(inicio, fin, etiqueta)})."""
    texto, ents, pos = "", set(), 0
    for m in MARCA.finditer(marcado):
        texto += marcado[pos:m.start()]
        ents.add((len(texto), len(texto) + len(m.group(1)), m.group(2)))
        texto += m.group(1)
        pos = m.end()
    return texto + marcado[pos:], ents

NLP = {idioma: cargar_modelo(nombre) for idioma, nombre in MODELOS.items()}
ETIQUETAS = ["PER", "ORG", "LOC", "MISC"]
print("spaCy", spacy.__version__, {i: (n.meta["name"], n.meta["version"]) for i, n in NLP.items()})
print("Modelos cargados correctamente.")

---
## 1. POS y dependencias

Un modelo de spaCy es una *pipeline*: tokeniza y luego añade, token a token, la categoría gramatical (`pos_`), el lema (`lemma_`), la relación sintáctica con su cabeza (`dep_`) y, más adelante, las entidades (`ents`). Vamos a mirar una frase corta en cada idioma.

In [ ]:
# datos-es
FRASE_DEMO = {
    "spa": "Lucía Sevilla presentó ayer el plan en Bilbao.",
    "ita": "Lucia Salerno ha presentato ieri il piano a Bologna.",
}
FRASE_DOS_VERBOS = {
    "spa": "El alcalde inauguró la feria y los vecinos visitaron el museo.",
    "ita": "Il sindaco ha inaugurato la fiera e i cittadini hanno visitato il museo.",
}

In [ ]:
def tabla_tokens(doc):
    return pd.DataFrame([{"i": t.i, "token": t.text, "pos": t.pos_, "lema": t.lemma_, "dep": t.dep_, "cabeza": t.head.text}
                         for t in doc])

docs_demo = {idioma: NLP[idioma](FRASE_DEMO[idioma]) for idioma in MODELOS}
for idioma, doc in docs_demo.items():
    print(idioma, "→", doc.text)
    display(tabla_tokens(doc))
print("nsubj =", spacy.explain("nsubj"), "| obj =", spacy.explain("obj"), "| ROOT =", spacy.explain("ROOT"))

# Comprobación robusta: el verbo principal tiene lema en infinitivo
raiz = {i: [t for t in d if t.dep_ == "ROOT"][0] for i, d in docs_demo.items()}
assert raiz["spa"].lemma_ == "presentar" and raiz["ita"].lemma_ == "presentare"
print("Lemas de la raíz:", {i: r.lemma_ for i, r in raiz.items()})

### ¿Qué observas?

- La **raíz** (`ROOT`) es el verbo. En italiano la forma compuesta `ha presentato` tiene un auxiliar (`aux`) y la raíz es el participio; en español el pretérito `presentó` es una sola palabra.
- Los nombres propios (`Lucía Sevilla`) forman un bloque: la segunda palabra cuelga de la primera con una relación `flat`.
- Es la base de todo lo que sigue: para «quién hace qué» buscaremos el hijo `nsubj` y el hijo `obj` de cada verbo.

### El árbol como estructura de datos

Cada token tiene `head` (su cabeza) y `children` (sus dependientes); `subtree` es el token con todo lo que cuelga de él. Imprimimos el árbol con sangría desde la raíz.

In [ ]:
def imprimir_arbol(token, nivel=0):
    print("    " * nivel + f"{token.text} [{token.dep_}]")
    for hijo in token.children:
        imprimir_arbol(hijo, nivel + 1)

for idioma, doc in docs_demo.items():
    print(idioma)
    imprimir_arbol(raiz[idioma])

# Propiedad del árbol: todo token está en el subárbol de la raíz de su frase
for idioma, doc in docs_demo.items():
    assert len(list(raiz[idioma].subtree)) == len(doc)

---
## 2. Extraer «quién hace qué» del árbol

La idea: para cada verbo, su sujeto es el hijo con `dep_` que empieza por `nsubj` y su objeto es el hijo `obj`. Devolvemos el texto del **subárbol** de cada uno, para conservar los determinantes y los nombres compuestos. Antes de fiarnos del modelo, probamos la función con un árbol **construido a mano** (`Doc` con cabezas y dependencias fijadas): así el `assert` comprueba nuestro código y no las previsiones del modelo.

In [ ]:
def extraer_svo(doc):
    """Lista de (sujeto, lema del verbo, objeto) para cada verbo que tenga ambos."""
    ternas = []
    for t in doc:
        if t.pos_ != "VERB":
            continue
        sujetos = [h for h in t.children if h.dep_.startswith("nsubj")]
        objetos = [h for h in t.children if h.dep_ == "obj"]
        if sujetos and objetos:
            sujeto = "".join(x.text_with_ws for x in sujetos[0].subtree).strip()
            objeto = "".join(x.text_with_ws for x in objetos[0].subtree).strip()
            ternas.append((sujeto, t.lemma_, objeto))
    return ternas

In [ ]:
# datos-es
from spacy.tokens import Doc
_v = spacy.blank("es").vocab
# Árbol escrito a mano (cabezas absolutas)
PALABRAS = ["La", "alcaldesa", "inauguró", "la", "feria", "y", "el", "concejal", "visitó", "el", "museo", "."]
doc_a_mano = Doc(_v, words=PALABRAS,
                 heads=[1, 2, 2, 4, 2, 8, 7, 8, 2, 10, 8, 2],
                 deps=["det", "nsubj", "ROOT", "det", "obj", "cc", "det", "nsubj", "conj", "det", "obj", "punct"],
                 pos=["DET", "NOUN", "VERB", "DET", "NOUN", "CCONJ", "DET", "NOUN", "VERB", "DET", "NOUN", "PUNCT"],
                 lemmas=["el", "alcaldesa", "inaugurar", "el", "feria", "y", "el", "concejal", "visitar", "el", "museo", "."])

In [ ]:
esperado = [("La alcaldesa", "inaugurar", "la feria"), ("el concejal", "visitar", "el museo")]     # dato-es
assert extraer_svo(doc_a_mano) == esperado, "extraer_svo no recupera las dos ternas del árbol escrito a mano"
print("Árbol a mano:", extraer_svo(doc_a_mano))

for idioma in MODELOS:
    doc = NLP[idioma](FRASE_DOS_VERBOS[idioma])
    print(idioma, FRASE_DOS_VERBOS[idioma])
    print("  ternas:", extraer_svo(doc))
    print("  verbos:", [(t.text, t.dep_, t.head.text) for t in doc if t.pos_ == "VERB"])
    print("  sujetos:", [(t.text, t.dep_, t.head.text) for t in doc if t.dep_.startswith("nsubj")])

### ¿Qué observas?

Con spaCy 3.8 y `es_core_news_sm` 3.8.0 se recuperan las dos ternas de la frase con dos verbos. Con `it_core_news_sm` 3.8.0 el árbol de esa misma frase cuelga el segundo sujeto (`cittadini`) como coordinado del objeto `fiera`, y el segundo verbo se queda **sin sujeto**: la función solo devuelve una terna. El extractor no falla: recibe un árbol equivocado. (Con otra versión del modelo el resultado puede cambiar.) Regla práctica: la extracción hereda los errores del análisis sintáctico, y un modelo pequeño se equivoca sobre todo con coordinaciones y frases largas.

---
## 3. NER y el corpus anotado

Para evaluar un NER hace falta una **referencia** (*gold*): frases con las entidades marcadas por una persona. Hemos escrito 67 frases de noticias sintéticas por idioma (personas inventadas; lugares y organizaciones genéricos o reales, sin noticias falsas) y las hemos anotado **a mano** con solo cuatro etiquetas (PER, ORG, LOC, MISC), las mismas que producen los modelos pequeños de spaCy. Cada fila de `noticias_ner_es.csv` y `noticias_ner_it.csv` tiene `id`, `texto` y `entidades` (JSON con `[inicio, fin, etiqueta]` en caracteres). Los criterios completos están en `datos/LEEME.md`.

In [ ]:
ARCHIVO_NER = {"spa": "noticias_ner_es.csv", "ita": "noticias_ner_it.csv"}      # dato-es

def cargar_corpus(idioma):
    """Lee el corpus NER anotado: lista de (id, texto, {(inicio, fin, etiqueta)})."""
    df = pd.read_csv(cargar_datos(ARCHIVO_NER[idioma]))
    return [(r.id, r.texto, {(a, b, e) for a, b, e in json.loads(r.entidades)}) for r in df.itertuples()]

CORPUS = {idioma: cargar_corpus(idioma) for idioma in MODELOS}
for idioma, corpus in CORPUS.items():
    for _, texto, ents in corpus:
        assert all(0 <= a < b <= len(texto) for a, b, _ in ents), "desplazamiento fuera del texto"
        assert all(e in ETIQUETAS for _, _, e in ents), "etiqueta desconocida"
    assert len(corpus) >= 60
GOLD = {i: [g for _, _, g in c] for i, c in CORPUS.items()}
TEXTOS = {i: [t for _, t, _ in c] for i, c in CORPUS.items()}
IDS = {i: [x for x, _, _ in c] for i, c in CORPUS.items()}
conteo = pd.DataFrame({i: Counter(e for g in GOLD[i] for _, _, e in g) for i in CORPUS}).loc[ETIQUETAS]
conteo.loc["total"] = conteo.sum()
print({i: len(c) for i, c in CORPUS.items()}, "frases por idioma")
conteo

In [ ]:
for idioma in MODELOS:
    doc = NLP[idioma](FRASE_DEMO[idioma])
    print(idioma, [(e.text, e.label_, e.start_char, e.end_char) for e in doc.ents])
# Las entidades de spaCy son tramos de caracteres: entidad = texto[inicio:fin]
doc = NLP["spa"](FRASE_DEMO["spa"])
assert all(doc.text[e.start_char:e.end_char] == e.text for e in doc.ents)

---
## 4. Evaluar un NER: primero, qué es un acierto

Antes de calcular nada hay que decidir **qué cuenta como acierto**. Una entidad predicha es un trío `(inicio, fin, etiqueta)`. Tres definiciones razonables:

| Modo | Es acierto (TP) si… | Cuándo sirve |
|---|---|---|
| `exacto` | el tramo **y** la etiqueta coinciden | el estándar; el más estricto |
| `tramo` | el tramo coincide, la etiqueta se ignora | medir solo la *localización* |
| `parcial` | misma etiqueta y los tramos se **solapan** | tolerar fronteras imprecisas |

Con cualquiera de ellas: **precisión** = TP/(TP+FP) (de lo que dijo el modelo, cuánto es correcto), ***recall*** = TP/(TP+FN) (de lo que había, cuánto encontró) y **F1** = media armónica. Una entidad predicha que no acierta es un FP y una entidad de la referencia no encontrada es un FN: **un error de frontera cuenta como FP y FN a la vez** en el modo exacto. Usaremos el modo `exacto` salvo que se diga lo contrario. Si un denominador es 0 (una etiqueta sin predicciones, por ejemplo), por convención el valor es 0.

In [ ]:
def solapan(a, b):
    """Dos tramos (inicio, fin, ...) se solapan si comparten al menos un carácter."""
    return a[0] < b[1] and b[0] < a[1]

def coincide(a, b, modo):
    if modo == "exacto":                       # mismo tramo y misma etiqueta
        return a == b
    if modo == "tramo":                        # mismo tramo, etiqueta ignorada
        return a[:2] == b[:2]
    return a[2] == b[2] and solapan(a, b)      # "parcial": misma etiqueta y tramos solapados

def contar(gold, pred, modo="exacto"):
    """Devuelve {etiqueta: [tp, fp, fn]}. gold y pred: una lista de conjuntos {(inicio, fin, etiqueta)} por frase."""
    cuenta = {e: [0, 0, 0] for e in ETIQUETAS}
    for g, p in zip(gold, pred):
        g, p = sorted(g), sorted(p)
        usados_g, usados_p = set(), set()
        for i, a in enumerate(g):
            for j, b in enumerate(p):
                if j not in usados_p and coincide(a, b, modo):
                    usados_g.add(i); usados_p.add(j); cuenta[a[2]][0] += 1
                    break
        for j, b in enumerate(p):
            if j not in usados_p:
                cuenta[b[2]][1] += 1
        for i, a in enumerate(g):
            if i not in usados_g:
                cuenta[a[2]][2] += 1
    return cuenta

def prf(tp, fp, fn):
    """Precisión, recall y F1; si el denominador es 0, el valor es 0."""
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

def resumen(cuenta):
    tp, fp, fn = (sum(v[k] for v in cuenta.values()) for k in range(3))
    return prf(tp, fp, fn), (tp, fp, fn)

def tabla(cuenta):
    filas = []
    for e in ETIQUETAS:
        tp, fp, fn = cuenta[e]
        filas.append([e, tp, fp, fn, *prf(tp, fp, fn)])
    (p, r, f), (tp, fp, fn) = resumen(cuenta)
    filas.append(["TOTAL (micro)", tp, fp, fn, p, r, f])
    return pd.DataFrame(filas, columns=["etiqueta", "tp", "fp", "fn", "precisión", "recall", "F1"]).round(3)

### Calcular a mano un ejemplo pequeño

Cuatro frases; para cada una, la referencia y la predicción de un modelo **imaginario** (con `{texto|ETIQUETA}` marcamos las entidades). Cuenta antes de ejecutar: en el modo exacto hay 4 aciertos (`Ana Ruiz`, `Bilbao`, `Ginebra`, `Roma`), 3 FP (`ONU` con etiqueta equivocada, `Cruz` con frontera equivocada, `Ayer`) y 3 FN (`ONU` como ORG, `Cruz Roja`, `italianos`). Por tanto P = 4/7, R = 4/7 y F1 = 4/7 ≈ 0,571.

In [ ]:
# datos-es
# Ejemplo mínimo para calcular a mano: cada frase con la referencia (gold) y la predicción de un modelo imaginario
MANO = {
    "spa": [("Ana Ruiz vive en Bilbao.", "{Ana Ruiz|PER} vive en {Bilbao|LOC}.", "{Ana Ruiz|PER} vive en {Bilbao|LOC}."),
            ("La ONU se reunió en Ginebra con la Cruz Roja.", "La {ONU|ORG} se reunió en {Ginebra|LOC} con la {Cruz Roja|ORG}.", "La {ONU|LOC} se reunió en {Ginebra|LOC} con la {Cruz|ORG} Roja."),
            ("Los italianos llegaron ayer.", "Los {italianos|MISC} llegaron ayer.", "Los italianos llegaron ayer."),
            ("Ayer llovió en Roma.", "Ayer llovió en {Roma|LOC}.", "{Ayer|PER} llovió en {Roma|LOC}.")],
    "ita": [("Anna Rossi vive a Bologna.", "{Anna Rossi|PER} vive a {Bologna|LOC}.", "{Anna Rossi|PER} vive a {Bologna|LOC}."),
            ("L'ONU si è riunita a Ginevra con la Croce Rossa.", "L'{ONU|ORG} si è riunita a {Ginevra|LOC} con la {Croce Rossa|ORG}.", "L'{ONU|LOC} si è riunita a {Ginevra|LOC} con la {Croce|ORG} Rossa."),
            ("Gli italiani sono arrivati ieri.", "Gli {italiani|MISC} sono arrivati ieri.", "Gli italiani sono arrivati ieri."),
            ("Ieri è piovuto a Roma.", "Ieri è piovuto a {Roma|LOC}.", "{Ieri|PER} è piovuto a {Roma|LOC}.")],
}

In [ ]:
for idioma in MODELOS:
    gold_m, pred_m = [], []
    for _, ref, pred in MANO[idioma]:
        t1, g = parsear(ref)
        t2, p = parsear(pred)
        assert t1 == t2, "la referencia y la predicción deben tener el mismo texto"
        gold_m.append(g); pred_m.append(p)
    resultados = {modo: resumen(contar(gold_m, pred_m, modo)) for modo in ("exacto", "tramo", "parcial")}
    # Cifras calculadas a mano en el texto
    assert resultados["exacto"][1] == (4, 3, 3)
    assert np.isclose(resultados["exacto"][0][2], 4 / 7)
    assert resultados["parcial"][1] == (5, 2, 2) and resultados["tramo"][1] == (5, 2, 2)
    cuenta = contar(gold_m, pred_m)
    assert cuenta["PER"] == [1, 1, 0] and cuenta["LOC"] == [3, 1, 0] and cuenta["ORG"] == [0, 1, 2] and cuenta["MISC"] == [0, 0, 1]
    print(idioma, {m: (r[1], tuple(round(x, 3) for x in r[0])) for m, r in resultados.items()})
tabla(contar(gold_m, pred_m))

### ¿Qué observas?

- La misma predicción da **F1 = 0,571 en modo exacto** y valores más altos en `tramo` y `parcial`: elegir el modo es una decisión de evaluación, y hay que decirla junto a la cifra.
- La etiqueta `ORG` tiene precisión y *recall* 0 (ningún acierto), y `MISC` tiene precisión 0 por convención (0/0) y *recall* 0: por eso conviene mirar **por etiqueta** y no solo el total.
- El total «micro» suma TP, FP y FN de todas las etiquetas antes de dividir; la media «macro» promediaría los F1 por etiqueta y da el mismo peso a etiquetas raras.

---
## 5. Experimento guiado: ¿qué tal lo hace el modelo pequeño?

Pasamos las 67 frases de cada idioma por su modelo y comparamos con la referencia, en los tres modos y por etiqueta. **Predice antes de ejecutar:** ¿qué etiqueta crees que será la más difícil? ¿En qué modo será más alta la F1?

In [ ]:
PRED = {i: [{(e.start_char, e.end_char, e.label_) for e in doc.ents} for doc in NLP[i].pipe(TEXTOS[i])] for i in MODELOS}
resultados = {}
for idioma in MODELOS:
    for modo in ("exacto", "tramo", "parcial"):
        resultados[idioma, modo] = contar(GOLD[idioma], PRED[idioma], modo)

n_gold = {i: sum(len(g) for g in GOLD[i]) for i in MODELOS}
for idioma in MODELOS:
    tp, fp, fn = resumen(resultados[idioma, "exacto"])[1]
    assert tp + fn == n_gold[idioma], "TP + FN debe ser el número de entidades de la referencia"
    assert tp + fp == sum(len(p) for p in PRED[idioma]), "TP + FP debe ser el número de entidades predichas"
    assert resumen(resultados[idioma, "exacto"])[1][0] <= resumen(resultados[idioma, "parcial"])[1][0] <= n_gold[idioma]

comparacion = pd.DataFrame({(idioma, modo): dict(zip(["precisión", "recall", "F1"], resumen(resultados[idioma, modo])[0]))
                            for idioma in MODELOS for modo in ("exacto", "tramo", "parcial")}).T.round(3)
display(comparacion)
for idioma in MODELOS:
    print(idioma, "modo exacto, por etiqueta")
    display(tabla(resultados[idioma, "exacto"]))

In [ ]:
f1_et = pd.DataFrame({idioma: [prf(*resultados[idioma, "exacto"][e])[2] for e in ETIQUETAS] for idioma in MODELOS}, index=ETIQUETAS)
ax = f1_et.plot.bar(rot=0, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("F1 (modo exacto)")
ax.set_ylim(0, 1)
ax.set_title("F1 por etiqueta con spaCy 3.8 (ejemplo ilustrativo)")
ax.legend(["es_core_news_sm", "it_core_news_sm"])
plt.show()
assert f1_et.shape == (4, 2) and ((f1_et >= 0) & (f1_et <= 1)).all().all()

### ¿Qué observas?

Con spaCy 3.8 y los modelos 3.8.0, sobre este corpus (*ejemplo ilustrativo*, F1 micro en modo exacto: unos 0,70 en español y 0,76 en italiano):

- La F1 sube bastante al pasar de `exacto` a `tramo` (≈ 0,90 y ≈ 0,88): **muchas entidades se localizan bien pero con otra etiqueta**.
- `PER` y `LOC` salen mejor que `ORG` y `MISC`; `ORG` tiene precisión alta y *recall* bajo: el modelo no la predice casi nunca donde debería.
- La diferencia entre idiomas viene de dos modelos distintos y de dos corpus que son adaptaciones, no traducciones exactas: **no se puede decir que el italiano sea «más fácil»**.
- Con 144 entidades por idioma, una sola entidad mueve la F1 en torno a 0,7 puntos porcentuales: las diferencias pequeñas no son significativas.

---
## 6. Errores típicos de un modelo pequeño

Clasificamos cada error del modo exacto en cuatro tipos:

| Tipo | Significado |
|---|---|
| `etiqueta` | mismo tramo, etiqueta distinta (p. ej. ORG → LOC) |
| `frontera` | el tramo se solapa con otro pero no coincide |
| `omitida` | entidad de la referencia sin ninguna predicción que se solape |
| `espuria` | predicción que no se solapa con ninguna entidad de la referencia |

In [ ]:
def tipos_error(gold, pred):
    tipos = Counter()
    for g, p in zip(gold, pred):
        for a in g - p:                                    # entidades de la referencia no acertadas
            if any(a[:2] == b[:2] for b in p):
                tipos["etiqueta"] += 1
            elif any(solapan(a, b) for b in p):
                tipos["frontera"] += 1
            else:
                tipos["omitida"] += 1
        for b in p - g:                                    # predicciones que no están en la referencia
            if not any(solapan(a, b) for a in g):
                tipos["espuria"] += 1
    return tipos

errores = {i: tipos_error(GOLD[i], PRED[i]) for i in MODELOS}
for idioma in MODELOS:
    fn = resumen(resultados[idioma, "exacto"])[1][2]
    assert errores[idioma]["etiqueta"] + errores[idioma]["frontera"] + errores[idioma]["omitida"] == fn, "los tres tipos deben sumar los FN"
display(pd.DataFrame(errores).fillna(0).astype(int).reindex(["etiqueta", "frontera", "omitida", "espuria"]))

confusion = {i: Counter((a[2], b[2]) for g, p in zip(GOLD[i], PRED[i]) for a in g for b in p if a[:2] == b[:2] and a[2] != b[2]) for i in MODELOS}
for idioma in MODELOS:
    print(idioma, "(etiqueta correcta → predicha):", confusion[idioma].most_common(4))

In [ ]:
# Casos difíciles del corpus: nombre de persona que coincide con un lugar, siglas y gentilicios en minúscula
def ver(idioma, prefijo_id):
    i = next(k for k, x in enumerate(IDS[idioma]) if x.endswith(prefijo_id))
    print(TEXTOS[idioma][i])
    print("   gold:", [(TEXTOS[idioma][i][a:b], e) for a, b, e in sorted(GOLD[idioma][i])])
    print("   spaCy:", [(TEXTOS[idioma][i][a:b], e) for a, b, e in sorted(PRED[idioma][i])])

for idioma in MODELOS:
    for pid in ("006", "010", "016"):
        ver(idioma, pid)
    print()

### ¿Qué observas?

- El error más frecuente en ambos idiomas es de **etiqueta**, sobre todo ORG → LOC: los modelos pequeños tratan como lugar una organización con nombre de aspecto geográfico o desconocido (`Cooperativa Mar Norte`, `Hospital Clínico Universitario`).
- Los **gentilicios en minúscula** (`italianos`, `portugueses`) suelen quedar sin marcar: la mayúscula inicial es una pista fuerte para estos modelos.
- Palabras con mayúscula por estar a principio de frase (`Según`, `Ayer`, `Ieri`) generan alguna entidad espuria.
- Otros errores de frontera: el modelo incluye una palabra genérica (`río Ebro`, `puerto de Valencia`) o corta un nombre largo.

- Los casos ambiguos de persona/lugar (`Carlos Toledo viajó a Toledo`) los resuelve bien este modelo: la dificultad está en otro sitio.

No son «fallos raros»: son la señal de qué reglas escribir a continuación.

---
## 7. Reglas que corrigen al modelo: `Matcher` y `EntityRuler`

- El **`Matcher`** busca **patrones de tokens** (`LOWER`, `IS_TITLE`, `POS`, `OP`…) y devuelve tramos; no cambia las entidades por sí solo.
- El **`EntityRuler`** es un componente que **asigna entidades** con patrones; colocado después del `ner` y con `overwrite_ents=True`, sus reglas mandan sobre el modelo.

Primero el `Matcher` con un patrón de «cargo + nombre propio» que usa el POS del modelo.

In [ ]:
# datos-es
CARGOS = {"spa": ["alcalde", "alcaldesa", "doctora", "doctor", "presidente", "periodista", "actriz", "chef", "entrenador"],
          "ita": ["sindaco", "dottoressa", "dottore", "presidente", "giornalista", "attrice", "chef", "direttore"]}

In [ ]:
from spacy.matcher import Matcher
from spacy.util import filter_spans

def buscar_cargos(nlp, texto, cargos):
    matcher = Matcher(nlp.vocab)
    matcher.add("CARGO_NOMBRE", [[{"LOWER": {"IN": cargos}}, {"POS": "PROPN", "OP": "+"}]])
    doc = nlp(texto)
    return [s.text for s in filter_spans([doc[a:b] for _, a, b in matcher(doc)])]        # filter_spans se queda con el tramo más largo

for idioma in MODELOS:
    hallazgos = [(x, buscar_cargos(NLP[idioma], t, CARGOS[idioma])) for x, t in zip(IDS[idioma], TEXTOS[idioma])]
    hallazgos = [h for h in hallazgos if h[1]]
    print(idioma, len(hallazgos), "frases con «cargo + nombre»; ejemplos:", hallazgos[:3])

# Comprobación con una frase escrita para el test: la regla usa POS, así que solo comprobamos que el Matcher devuelve tramos de texto
_t = buscar_cargos(NLP["spa"], "El alcalde Pedro Soler habló ayer.", CARGOS["spa"])       # dato-es
assert all(isinstance(x, str) for x in _t)

### Un `EntityRuler` con reglas de desarrollo

Para no engañarnos, dividimos cada corpus: las **40 primeras frases** son de *desarrollo* (miramos sus errores y escribimos reglas) y las **27 restantes**, de *prueba* (no las tocamos al escribir las reglas). Las reglas son tres: siglas de organizaciones, gentilicios y un patrón `ORG` = nombre de institución con mayúscula + nombre propio (con posibles preposiciones). Las listas salen de los errores de desarrollo.

In [ ]:
# datos-es
SIGLAS = {"spa": ["ONU", "FMI", "OMS", "UGT"], "ita": ["ONU", "FMI", "OMS", "CGIL"]}
GENTILICIOS = {"spa": ["italianos", "portugueses", "gallega"], "ita": ["italiani", "portoghesi", "ligure"]}
NOMBRES_ORG = {"spa": ["cooperativa", "banco", "universidad", "hospital", "comisión", "gobierno", "federación", "fundación", "ayuntamiento", "óptica"],
               "ita": ["cooperativa", "banca", "università", "ospedale", "commissione", "regione", "federazione", "fondazione", "comune", "politecnico", "ottica"]}
PREPOSICIONES = ["de", "del", "la", "di", "della", "dei", "dell'"]

In [ ]:
def patrones(idioma):
    nombre = {"LOWER": {"IN": NOMBRES_ORG[idioma]}, "IS_TITLE": True}
    titulo = {"IS_TITLE": True, "OP": "+"}
    prep = {"LOWER": {"IN": PREPOSICIONES}}
    pats = [{"label": "ORG", "pattern": [{"ORTH": s}]} for s in SIGLAS[idioma]]
    pats.append({"label": "MISC", "pattern": [{"LOWER": {"IN": GENTILICIOS[idioma]}}]})
    for esquema in ([nombre, titulo], [nombre, prep, titulo], [nombre, titulo, prep, titulo], [nombre, prep, titulo, prep, titulo]):
        pats.append({"label": "ORG", "pattern": esquema})
    return pats

def con_reglas(idioma):
    nlp = spacy.load(MODELOS[idioma])
    ruler = nlp.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})
    ruler.add_patterns(patrones(idioma))
    return nlp

NLP_R = {i: con_reglas(i) for i in MODELOS}
print({i: n.pipe_names for i, n in NLP_R.items()})

# Comprobación robusta: las reglas mandan sobre el modelo
e_es = {e.text: e.label_ for e in NLP_R["spa"]("La ONU y los italianos llegaron a la reunión.").ents}       # dato-es
e_it = {e.text: e.label_ for e in NLP_R["ita"]("L'ONU e gli italiani sono arrivati alla riunione.").ents}      # dato-es
assert e_es["ONU"] == "ORG" and e_es["italianos"] == "MISC"
assert e_it["ONU"] == "ORG" and e_it["italiani"] == "MISC"

In [ ]:
DESARROLLO = slice(0, 40)
PRUEBA = slice(40, None)
PRED_R = {i: [{(e.start_char, e.end_char, e.label_) for e in doc.ents} for doc in NLP_R[i].pipe(TEXTOS[i])] for i in MODELOS}

filas = []
for idioma in MODELOS:
    for nombre, parte in (("desarrollo", DESARROLLO), ("prueba", PRUEBA)):
        for sistema, pred in (("solo modelo", PRED), ("modelo + reglas", PRED_R)):
            (p, r, f), (tp, fp, fn) = resumen(contar(GOLD[idioma][parte], pred[idioma][parte]))
            assert tp + fn == sum(len(g) for g in GOLD[idioma][parte])
            filas.append([idioma, nombre, sistema, tp, fp, fn, round(p, 3), round(r, 3), round(f, 3)])
pd.DataFrame(filas, columns=["idioma", "partición", "sistema", "tp", "fp", "fn", "precisión", "recall", "F1"])

### ¿Qué observas?

Con spaCy 3.8 (*ejemplo ilustrativo*): las reglas suben la F1 en **desarrollo** (de ≈ 0,70 a ≈ 0,80 en español y de ≈ 0,70 a ≈ 0,82 en italiano), pero en **prueba** no se nota ninguna mejora (las cifras de antes y después no cambian): las listas salen de lo que vimos en desarrollo (`ONU`, `italianos`…) y en prueba aparecen otras siglas y gentilicios (`UE`, `catalanes`…). Es el riesgo de escribir reglas mirando los errores: **una mejora en las frases usadas para escribirlas no dice nada de las demás**. Hay que medir siempre en frases apartadas, ampliar las reglas con criterio (o con más datos) y comprobar que no rompen aciertos previos.

---
## Resumen

| Concepto | Idea clave |
|---|---|
| POS y dependencias | el árbol conecta cada palabra con su cabeza; `nsubj` y `obj` dan «quién hace qué» |
| Extracción SVO | hereda los errores del análisis; comprobar el código con un árbol escrito a mano |
| NER | entidades como tramos `(inicio, fin, etiqueta)` de caracteres |
| Evaluación | definir antes el acierto (exacto, tramo, parcial); P, R y F1 **por etiqueta** y en total |
| Errores típicos | etiqueta (ORG → LOC), frontera, omisión y entidades espurias |
| Reglas | `Matcher` busca; `EntityRuler` asigna; medir en frases no usadas para escribirlas |

**Siguiente:** en la clase 8 salimos de la estructura y estudiamos los **modelos de lenguaje n-grama**: cómo asignar probabilidad a una secuencia de palabras. Después, la práctica de esta clase.